### Improving language understanding by generative pre-training

**Model input**: NxE matrix of token embeddings
- N = context window size, E = embedding dimension
- There's no start token but the context window may include padding tokens and
  an end of sequence token.
- Context windows are slid over the token corpus.
---
**Model**: Transformer decoder (the masked self attention adds auto-regressivity).
- During inference, the model receives an ever growing chunk of tokens and outputs
  a single token in aloop. This is inherently sequential 😬.

- Training is split into two phases:
  - **Self supervised pre-training**: The model receives a chunk of tokens and
    outputs a probability over a chunk of tokens. We want to maximize the probability
    of outputting the next token in the sequence (the shifted context window).

  - **Supervised fine-tuning**: The model receives a chunk of tokens and outputs
    a probability over an output label *y*. We want to maximize the probability
    of outputting the correct label and the next correct output token. The model
    parameters are transferred and the last linear layer is swapped out.
---
**Experiment ideas**:
- Remove the fine tuning stage and go diretly to text generation
- Overfit on a tiny dataset to make sure that the model is implemented correctly
- Compare validation losses:
  - Different context window sizes
  - Different number of layers
  - Different number of attention heads
  - Different activation functions (what is GELU even doing?)
  - Remove/replace LayerNorm
  - Remove causal mask
  - Different dataset sizes
  - Different parameter counts
  - Vary temperature and experiment with top-k, top-p, greedy decoding, etc
  - What if we remove the MLP?
  - What if we remove attention?
---
Datasets:
- [wikitext](https://huggingface.co/datasets/Salesforce/wikitext)
- [OpenPhi textbooks](https://huggingface.co/datasets/open-phi/textbooks)
- [Youtube comment sentiment](https://huggingface.co/datasets/AmaanP314/youtube-comment-sentiment)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import datasets
import tiktoken
import math

def preprocess(rows, enc):
  tokens = [enc.encode(f"{r}<EOS>") for r in rows["markdown"]]
  return {"token_ids": tokens, "len": len(tokens)}

def postprocess(dataset, ctx_win_size):
  lengths = torch.tensor(dataset["len"], dtype=torch.int32)
  max_len = torch.max(lengths).item()
  padded_len = ctx_win_size * math.ceil(max_len / ctx_win_size)

  tokens = []
  for row in dataset["token_ids"]:
    tensor = torch.tensor(row, dtype=torch.float32)
    pad = padded_len - tensor.numel()
    tokens.append(F.pad(tensor, (0, pad), value=0))

  chunks = torch.split(torch.stack(tokens), ctx_win_size, dim=1)
  return chunks, lengths

def load_dataset(name, cache_path, ctx_win_size):
  try:
    data = torch.load(cache_path)
    return data["chunks"], data["lengths"]
  except:
    base = tiktoken.get_encoding("gpt2")
    enc = tiktoken.Encoding(
      "got2",
      pat_str=base._pat_str,
      mergeable_ranks=base._mergeable_ranks,
      special_tokens={"<EOS>": base.max_token_value},
    )
    dataset = datasets.load_dataset(name, split="train")
    dataset = dataset.map(lambda rows: preprocess(rows, enc), batched=True)
    chunks, lengths = postprocess(dataset, ctx_win_size)
    torch.save({"chunks": chunks, "lengths": lengths}, cache_path)
    return chunks, lengths

torch.manual_seed(67)
ctx_win_size = 1024
token_chunks, seq_lengths = \
  load_dataset("open-phi/textbooks", ".cache/textbooks.pth", ctx_win_size)


In [ ]:
class Decoder(nn.Module):
  def __init__(self, emebed_dim, mlp_dim, num_heads, dropout):
    super().__init__()
    self.mha = nn.MultiHeadAttention(embed_dim, num_heads, dropout=dropout)
    self.norm1 = nn.LayerNorm(embed_dim)
    self.mlp = nn.Sequential(
      nn.Linear(embed_dim, mlp_dim),
      nn.GELU(),
      nn.Linear(mlp_dim, embed_dim)
    )
    self.norm2 = nn.LayerNorm(embed_dim)

  def forward(self, x, pad_mask, mask):
    x1, _ = self.mha(x, x, x, attn_mask=mask, key_padding_mask=pad_mask)
    x2 = self.norm1(x + x1)
    x3 = self.mlp(x2)
    x4 = self.norm2(x3 + x2)
    return x4
